Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef
)

import joblib


Load Dataset

Removes missing values

Final dataset = 30,162 rows - Below

In [2]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
df = pd.read_csv(url, header=None, names=columns, skipinitialspace=True)

df.replace("?", pd.NA, inplace=True)
df.dropna(inplace=True)

df.shape


(30162, 15)

Split Features & Target

In [3]:
X = df.drop("income", axis=1)
y = df["income"]


In [9]:
y = y.map({"<=50K": 0, ">50K": 1})

Train/Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


Column Types

Which columns are numeric

Which columns are categorical

Preprocessing Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# # Separate features and target FIRST
# X = df.drop("income", axis=1)
# y = df["income"]

# Identify column types from X (NOT df)
categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ],
    sparse_threshold=0  
)


In [19]:
print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)


Categorical columns: Index(['workclass', 'education', 'marital_status', 'occupation',
       'relationship', 'race', 'sex', 'native_country'],
      dtype='object')
Numerical columns: Index(['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss',
       'hours_per_week'],
      dtype='object')


Defining Models

In [20]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}


Training, Evaluating & Saving

In [21]:
results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

    joblib.dump(
        pipeline,
        f"./models/{name.replace(' ', '_').lower()}.pkl"
    )


View Results

In [22]:
results_df = pd.DataFrame(results)
results_df


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.847505,0.902198,0.735437,0.605193,0.663988,0.571063
1,Decision Tree,0.813360,0.753571,0.622876,0.634487,0.628628,0.504040
2,KNN,0.826952,0.859544,0.666424,0.610519,0.637248,0.524774
3,Naive Bayes,0.582629,0.801815,0.367846,0.941411,0.528994,0.364323
4,Random Forest,0.849494,0.900373,0.729876,0.627830,0.675018,0.580582


Install XGBoost and Import

In [24]:
%pip install xgboost
from xgboost import XGBClassifier


Note: you may need to restart the kernel to use updated packages.


Adding it to your models dictionary

In [25]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}


Train & Evaluate

In [ ]:
results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        auc = None

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": auc,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

    joblib.dump(
        pipeline,
        f"./models/{name.replace(' ', '_').lower()}.pkl"
    )


/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [00:10:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Checking Results

In [27]:
results_df = pd.DataFrame(results)
results_df


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.847505,0.902198,0.735437,0.605193,0.663988,0.571063
1,Decision Tree,0.813360,0.753571,0.622876,0.634487,0.628628,0.504040
2,KNN,0.826952,0.859544,0.666424,0.610519,0.637248,0.524774
3,Naive Bayes,0.582629,0.801815,0.367846,0.941411,0.528994,0.364323
4,Random Forest,0.849494,0.900373,0.729876,0.627830,0.675018,0.580582
5,XGBoost,0.863252,0.922683,0.766745,0.647803,0.702274,0.618006
